**Paper Title: Revealing Land Use Dynamics in Armed-Conflict Hotspots in North-East Nigeria Using Earth Observation Data**

**Citation** : APA STYLE: Lateef, L. O., Tella A.,  Miano J. R., & Aina Y. A. (2025). Revealing Land Use Dynamics in Armed-Conflict Hotspots in North-East Nigeria Using Earth Observation Data. Land Use Policy, 157, 107673. https://doi.org/10.1016/j.landusepol.2025.107673

In [11]:
# Import libraries
import datetime
import time
import os
from pathlib import Path
import glob
from glob import glob
from typing import Any

import ee
import geemap
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv
from dtmapi import DTMApi
import osmnx as ox


import pandas as pd
import geopandas as gpd
import numpy as np
from scipy import stats
import pyproj
from shapely.geometry import Point, Polygon
import rasterio as rio

import folium
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# display all floating-point numbers with 3 decimal places
pd.set_option("display.float_format", "{:.3f}".format)

In [ ]:
"""
If you encountered this error: "CRSError: Invalid projection: EPSG:4326: (Internal Proj Error: proj_create: no database context specified)"
That is because pyproj is attempting to get the projection from a database folder different from your Python env folder.
Run the code snippet in this cell to resolve that.
If you use Windows OS, you can add the python env directory to your system environmental variable.
PROJ_LIB
C:\ProgramData\miniconda3\envs\land-conflict\Library\share\proj
"""
"""
# To get data directory being used for PROJ database
print(pyproj.datadir.get_data_dir())

# Then Set the directory to your python env directory
python_env_path= str(pyproj.datadir.get_data_dir())
python_env_path= "C:/ProgramData/miniconda3/envs/land-conflict/Library/share/proj"
pyproj.datadir.set_data_dir(python_env_path)
"""

In [12]:
load_dotenv()

True

In [13]:
# Initialize the library
gee_project_name = os.getenv("GEE_PROJECT_NAME")

# Trigger the authentication flow 
try:
    ee.Initialize(project= gee_project_name)
except:
    ee.Authenticate()
    ee.Initialize(project= gee_project_name) 

## . Parameters Definition

In [ ]:
# Path to ACLED data
acled_nga_data_path = "Data/ACLED/1997-01-01-2022-12-31-Nigeria.csv"

# Nigeria admin. level 1 boundary 
nga_bdry_shp_path = "Data/Boundary/GADM/gadm41_NGA_shp/gadm41_NGA_1.shp"

# ROIs boundary shapefile
grid3_settlements_path = "Data/Boundary/GRID_3/GRID_3_NGA_Settlement_Extents_Selected.shp"

# Folder for all GEE ouputs
out_google_folder = "Land_Conflicts_Nigeria"

# Default map centre coordinates
map_centre_coord = (11.10, 11.38)

## 1. Visualization Functions

In [ ]:
def feature_map(
    feature: ee.FeatureCollection,
    vis_params: dict[str, Any],
    name: str,
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Render an Earth Engine FeatureCollection on a satellite basemap."""

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.add_basemap("SATELLITE")
    styled_layer = feature.style(**vis_params)
    Map.add_layer(styled_layer, {}, name)
    return Map


def geodata_map(
    gdf: gpd.GeoDataFrame,
    style: dict[str, Any],
    name: str,
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Render a GeoDataFrame as a styled vector layer on a satellite basemap."""

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.add_basemap("SATELLITE")
    Map.add_gdf(gdf, layer_name=name, style=style)
    return Map


def raster_map(
    raster: ee.Image,
    viz_params: dict[str, Any],
    name: str,
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Render an Earth Engine Image on a satellite basemap.

    Args:
        raster: ee.Image to display.
        viz_params: Visualisation parameters (e.g. bands, min, max, palette).
        name: Layer label shown in the map legend.
        centre_coord: Map centre as [latitude, longitude].
        zoom: Initial zoom level (default 8).

    Returns:
        Configured map with the raster layer added.
    """

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.add_basemap("SATELLITE")
    Map.add_layer(raster, viz_params, name)
    return Map


def slider_map(
    layer1: tuple[ee.ComputedObject, dict[str, Any], str],
    layer2: tuple[ee.ComputedObject, dict[str, Any], str],
    centre_coord: list[float],
    zoom: int = 8,
) -> geemap.Map:
    """Display two Earth Engine layers in a side-by-side split-panel map.

    Args:
        layer1: Left panel as (ee_object, vis_params, name).
        layer2: Right panel as (ee_object, vis_params, name).
        centre_coord: Map centre as [latitude, longitude].
        zoom: Initial zoom level (default 8).

    Returns:
        Configured map with a horizontal split slider.
    """

    ee_object1, vis_params1, name1 = layer1
    ee_object2, vis_params2, name2 = layer2

    left_layer = geemap.ee_tile_layer(ee_object1, vis_params1, name=name1)
    right_layer = geemap.ee_tile_layer(ee_object2, vis_params2, name=name2)

    Map = geemap.Map(center=centre_coord, zoom=zoom)
    Map.split_map(left_layer, right_layer)
    return Map

In [ ]:
def build_group_bar(
    data: pd.DataFrame,
    x_column: str,
    y_columns: list[str],
    bar_title: str,
    axis_x: str,
) -> None:
    """Build and display a grouped bar chart with two series.

    Args:
        data: Source DataFrame.
        x_column: Column name for the x-axis categories.
        y_columns: Two-element list of column names for the bar series.
        bar_title: Chart title.
        axis_x: x-axis label.
    """

    fig = px.bar(
        data_frame=data,
        x=x_column,
        y=y_columns,
        barmode="group",
        color_discrete_map={y_columns[0]: "orange", y_columns[1]: "red"},
    )

    fig.update_layout(
        title={
            #"text": bar_title,
            "text": f"<b>{bar_title}</b>",
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
        },
        title_font=dict(size=22, family="Arial", color="black"),
        xaxis_title=axis_x,
        yaxis_title="Frequency [count]",
        xaxis=dict(tickangle=-45),
        yaxis=dict(title_font=dict(size=14)),
        legend={"title": "Category"},
        height=700,
    )

    return fig

In [ ]:
def plot_line_chart_roi(
    data_frame: pd.DataFrame,
    x_column: str,
    y_columns: str | list[str],
    chart_title: str,
    axis_x: str,
    axis_y: str = "Frequency [Count]",
    fill_na: float | None = None,
) -> None:
    """Plot a multi-series line chart showing temporal trends across ROIs.

    Args:
        data_frame: DataFrame with a "name" column used to colour each series.
        x_column: Column name for the x-axis (typically "year").
        y_columns: Column name(s) to plot on the y-axis.
        chart_title: Chart title.
        axis_x: x-axis label.
        axis_y: y-axis label. Defaults to "Frequency [Count]".
        fill_na: If provided, fills NaN values with this value before plotting.
    """
    if fill_na is not None:
        data_frame = data_frame.fillna(fill_na)

    fig = px.line(
        data_frame,
        x=x_column,
        y=y_columns,
        markers=True,
        color="name",
        labels={"value": axis_y, x_column: axis_x},
    )

    fig.update_layout(
        title={
            #"text": chart_title,
            "text": f"<b>{chart_title}</b>",
            "x": 0.5,
            "xanchor": "center",
            "yanchor": "top",
        },
        title_font=dict(size=18, family="Arial", color="black"),
        xaxis_title=axis_x,
        yaxis_title=axis_y,
        legend_title="Location",
    )

    return fig

In [ ]:
# Generate Monthly HeatMap 
def plot_heatmap(data, var_column, chart_title):

    plt.figure(figsize=(17, 9))

    heatmap_fig = sns.heatmap(
        data, cmap="YlOrRd", annot=True, fmt=".0f", linewidths=.5, cbar=reversed,
        cbar_kws = {"label": var_column},
        annot_kws= {"fontsize": 13, "fontweight": "bold"}
    )

   
    colorbar = heatmap_fig.collections[0].colorbar
    colorbar.set_label(var_column, fontsize=20, fontweight="bold")
    colorbar.ax.invert_yaxis()

    # Axis labels
    plt.title(chart_title, fontsize=16, fontweight= "bold") 
    plt.xlabel("Month", fontsize=18, fontweight="bold")
    plt.ylabel("Year", fontsize=18, fontweight = "bold")
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    # Invert bar 
    heatmap_fig.collections[0].colorbar.ax.invert_yaxis()

    return heatmap_fig

## 2. Export Functions

In [ ]:
def export_image_to_drive(
    image: ee.Image,
    extent: ee.Geometry,
    epsg: str,
    scale: int,
    suffix: str,
    export_folder: str = "Land_Conflicts_Nigeria",
) -> ee.batch.Task:
    """Export a single Earth Engine image to Google Drive.

    NOTE: Expects "roi_name" and "image_date" properties set on the image during
    pre-processing. Falls back to "system:time_start" for the date, then
    "UnknownDate" if neither property is present.

    Args:
        image: ee.Image to export.
        extent: Export region geometry.
        epsg: Target CRS (e.g. "EPSG:4326").
        scale: Pixel resolution in metres.
        suffix: Prefix string prepended to the output filename.
        export_folder: Google Drive destination folder.

    Returns:
        The initiated ee.batch.Task.
    """

    roi_name = image.get("roi_name").getInfo() if image.get("roi_name") else "UnknownROI"

    date_info = (
        image.get("image_date").getInfo()
        if image.get("image_date")
        else ee.Date(image.get("system:time_start")).format("YYYY").getInfo()
        if image.get("system:time_start")
        else "UnknownDate"
    )

    file_name = f"{suffix}_{roi_name}_{date_info}"

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=file_name,
        folder=export_folder,
        fileNamePrefix=file_name,
        scale=scale,
        region=extent,
        crs=epsg,
        maxPixels=1e13,
    )

    task.start()
    print(f"Exporting {file_name} to Google Drive...")

    return task


def export_fc_to_drive(
    feature_collection: ee.FeatureCollection,
    export_desc: str,
    export_folder: str = "Land_Use_Conflicts_Nigeria",
) -> ee.batch.Task:
    """Export an Earth Engine FeatureCollection to Google Drive as a shapefile.

    Args:
        feature_collection: ee.FeatureCollection to export.
        export_desc: Export task description and output filename.
        export_folder: Google Drive destination folder.

    Returns:
        The initiated ee.batch.Task.
    """

    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=export_desc,
        folder=export_folder,
        fileFormat="SHP",
    )

    task.start()

    return task

## 3. Armed Conflict Analyses


### 3A. Armed Conflict Helper Functions 

In [ ]:
def wrangle_acled_nga(data_path: str | Path) -> pd.DataFrame:
    """Load and filter ACLED conflict data for Nigeria.

    Retains only armed-clash battles between 2000 and 2022, drops records
    with a missing state ("admin1"), and parses event dates into year and
    month columns.

    Args:
        data_path: Path to the ACLED CSV file.

    Returns:
        Filtered DataFrame of armed-clash events.
    """

    df = pd.read_csv(data_path)
    df.dropna(subset=["admin1"], inplace=True)

    df = df[
        (df["event_type"] == "Battles")
        & (df["sub_event_type"] == "Armed clash")
        & (df["year"] >= 2000)
        & (df["year"] <= 2022)
    ]

    df["event_date"] = pd.to_datetime(df["event_date"], format="%d %B %Y")
    df["year"] = df["event_date"].dt.year
    df["month"] = df["event_date"].dt.month

    return df


In [ ]:
#
"""

# Authentication details

acled_api_key = os.getenv("ACLED_API_KEY")  # Availabe on your ACLED account
acled_email = os.getenv("ACLED_EMAIL") # Your email address that is linked to the ACLED account


api_key = ""
email = ""
start_date_acled = 2000-01-01
end_date_Acled = 2022-12-31


data_list = []
page = 1
limit = 5000  # Maximum records per API call

while True:
    url = (f"https://api.acleddata.com/acled/read?"
    f"key={api_key}&"
    f"email={email}&"
    "country=Nigeria&"
    f"event_date={start_date_acled}|{end_date_Acled}&"
    "event_date_where=BETWEEN&"
    "format=json&"
    f"limit={limit}"
    f"page={page}")

    response = requests.get(url, auth=HTTPBasicAuth(api_key, email))
    
    if response.status_code == 200:
        response_json = response.json()
        new_data = response_json.get("data", [])
        if not new_data:
            break  # No more data to fetch

        data_list.extend(new_data)
        print(f"Fetched page {page}, Total records: {len(data_list)}")

        page +=1  # Move to the next set of records
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        break

data = pd.DataFrame(data_list)

# 
print("The years in the dataset: ", data["year"].unique())
print(data.head())

"""
# 

### 3B. ACLED Download and Wrangle

In [ ]:
# Read and clean ACLED
nga_acled_data = wrangle_acled_nga(acled_nga_data_path)
nga_acled_data.info()
display(nga_acled_data.head())

# Export cleaned ACLED for heat map in QGIS
# nga_acled_data.to_csv("Outputs/ACLED/Spreadsheets/ACLED_nga_armed_agg_point_2000_2022.csv")

In [ ]:
# Convert 'nga_acled_data' to a GeoDataFrame
nga_acled_data_gdf = gpd.GeoDataFrame(
    nga_acled_data, 
    geometry = gpd.points_from_xy(nga_acled_data.longitude, 
                                  nga_acled_data.latitude)
)

nga_acled_data_gdf.crs = "EPSG:4326"

# Datetime columns to strings
nga_acled_data_gdf_map = nga_acled_data_gdf.copy()

datetime_cols = nga_acled_data_gdf_map.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns

for col in datetime_cols:
    nga_acled_data_gdf_map[col] = (
        nga_acled_data_gdf_map[col]
        .dt.strftime("%Y-%m-%d")
    )

nga_acled_data_gdf_map.explore()

### 3C. Temporal Variations of Armed Clashes & Fatalities in Nigeria (2000 - 2022)

In [ ]:
# Group the by 'year' and aggregate number of 'sub_event_type' and 'fatalities'  
nga_armed_fat_agg_yr = (nga_acled_data
                    .groupby("year")
                    .agg({"sub_event_type":"count", "fatalities":"sum"})
                    .reset_index()
                    .rename(columns={"sub_event_type": "armed clash"})
                    .sort_values(by="year")
)

nga_armed_fat_agg_yr.head()

In [ ]:
# Number of armed clash events and fatalities in each YEAR (2000 -2022) in Nigeria
armed_fat_year_bar_title = "Armed Clash Events and Fatalities in Nigeria (2000 - 2022)"

acled_nga_year_fig = build_group_bar(
    nga_armed_fat_agg_yr,
    x_column="year",
    y_columns=["armed clash", "fatalities"],
    bar_title=armed_fat_year_bar_title,
    axis_x="Year"
)
acled_nga_year_fig.show()


#acled_nga_year_fig.write_image("Maps_charts/Conflicts_and_Fat_NGA_Year_Bar.png")

In [ ]:
# Get years with the highest number of armed clash in Nigeria
def top_n_years(df: pd.DataFrame, 
                column: str, 
                n: int = 5) -> list[int]:
                """Return the n years with the highest values in the given column."""
                return df.sort_values(column, ascending=False)["year"].head(n).tolist()


fat_year_top_five = top_n_years(nga_armed_fat_agg_yr, "armed clash")
fat_year_top_five   = top_n_years(nga_armed_fat_agg_yr, "fatalities")

print(f"Highest numbers of armed clashes were in: {', '.join(map(str, fat_year_top_five))}")
print(f"Highest numbers of fatalities were in:    {', '.join(map(str, fat_year_top_five))}")

### 3D. Most Armed Clashes And Fatalities By States (2000 - 2022)

In [ ]:
# Group by "admin1" and aggregate number of 'sub_event_type' and 'fatalities' 
df_nga_armed_fat_agg_st = (nga_acled_data
                    .groupby("admin1")
                    .agg({"sub_event_type":"count", "fatalities":"sum"})
                    .reset_index()
                    .rename(columns={"sub_event_type": "armed clash"})
                    .sort_values(by="armed clash", ascending=False)

)

df_nga_armed_fat_agg_st.head(7)

In [ ]:
# Visualize  armed clash events and fatalities by States (2000 -2022) 
armed_fat_state_bar_title = "Total Armed Clash Events and Fatalities by States (2000 - 2022)" 
acled_nga_state_fig = build_group_bar(df_nga_armed_fat_agg_st, 
                                      "admin1", 
                                      ["armed clash", "fatalities"], 
                                      armed_fat_state_bar_title, 
                                      "State") 

#acled_nga_state_fig.write_image("Maps_charts/Conflicts_and_Fat_NGA_State_Bar.png")

### 3E. Spatial Visualization of Aggregated Armed Clashes And Fatalities in Nigeria

In [ ]:
def acled_spatial_join(
    shp_file: str | Path,
    acled_dataframe: pd.DataFrame,
) -> tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """Join a Nigeria admin-1 boundary shapefile with an ACLED DataFrame.

    Reconciles state name mismatches between the shapefile and ACLED data by
    aligning sorted name arrays, then merges on the corrected names. Returns
    both a polygon GeoDataFrame and a point GeoDataFrame (state centroids).

    Args:
        shp_file: Path to the admin-1 boundary shapefile.
        acled_dataframe: DataFrame with ACLED conflict data; must contain
            an "admin1" column.

    Returns:
        Tuple of (polygon GeoDataFrame, point GeoDataFrame).
    """

    gdf_nga_bdry = gpd.read_file(shp_file)

    # unique names of the states 
    state_name_gdf = np.sort(gdf_nga_bdry["NAME_1"].unique())
    state_name_acled = np.sort(acled_dataframe["admin1"].unique())

    corrected_state_names = {
        acled: gdf
        for gdf, acled in zip(state_name_gdf, state_name_acled)
        if gdf != acled
    }

    acled_dataframe["admin1"] = acled_dataframe["admin1"].replace(corrected_state_names)

    joined_gdf = gdf_nga_bdry.merge(
        acled_dataframe, how="left", left_on="NAME_1", right_on="admin1"
    )

    joined_gdf["centroid"] = joined_gdf.geometry.centroid
    joined_gdf_point = (
        gpd.GeoDataFrame(joined_gdf, geometry="centroid")
        .drop(columns="geometry")
    )

    return joined_gdf, joined_gdf_point




gdf_nga_armed_fat_agg_st, gdf_nga_armed_fat_agg_st_point = acled_spatial_join(
                                                                            nga_bdry_shp_path, 
                                                                            df_nga_armed_fat_agg_st
                                                                                    )
display(gdf_nga_armed_fat_agg_st.head())

# Export the GeoDataFrame
#(gdf_nga_armed_fat_agg_st[["COUNTRY", "ISO_1", "admin1", "armed clash", "fatalities", "centroid"]]
#.to_file("Outputs/ACLED/Shapefiles/ACLED_nga_armed_fat_agg_st_point_20_22.shp"))

In [ ]:
# categorization function
def categorize_values(value, bins):
    for i, bin_range in enumerate(bins):
        if bin_range[0] <= value <= bin_range[1]:
            return i
    return len(bins)

# Ranges/Classification of values
armed_clash_bins = [(0, 50), (51, 100), (101, 250), (251, 500), (501, 2500)]
fatalities_bins = [(0, 100), (101, 500), (501, 2000), (2001, 5000), (5001, 20000)]

# Categorize the data
gdf_nga_armed_fat_agg_st_point["armed_clash_category"] = gdf_nga_armed_fat_agg_st_point["armed clash"].apply(lambda x: categorize_values(x, armed_clash_bins))
gdf_nga_armed_fat_agg_st_point["fatalities_category"] = gdf_nga_armed_fat_agg_st_point["fatalities"].apply(lambda x: categorize_values(x, fatalities_bins))

# Marker sizes for each category
armed_clash_sizes = [25, 85, 105, 225, 500]
fatalities_sizes = [25, 85, 105, 225, 500]

# Plot the maps
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# Use Nigeria admin bdry for basemape
nga_shp = gpd.read_file(nga_bdry_shp_path )

# Plot base map
nga_shp.plot(ax=axs[0], color="black")
nga_shp.plot(ax=axs[1], color="lightgrey")

# Plot the distribution of armed clash events 
for i, size in enumerate(armed_clash_sizes):
    subset = gdf_nga_armed_fat_agg_st_point[gdf_nga_armed_fat_agg_st_point["armed_clash_category"] == i]
    subset.plot(ax=axs[0], color="orange", markersize=size, label=f"{armed_clash_bins[i][0]} - {armed_clash_bins[i][1]}", alpha=0.5)

axs[0].set_title("Armed Clash Events in Nigeria (2000-2022)")
axs[0].set_xlabel("Longitude")
axs[0].set_ylabel("Latitude")
axs[0].legend(title='Armed Clashes', loc="lower right")

# Plot the distribution of fatalities 
for i, size in enumerate(fatalities_sizes):
    subset = gdf_nga_armed_fat_agg_st_point[gdf_nga_armed_fat_agg_st_point["fatalities_category"] == i]
    subset.plot(ax=axs[1], color="red", markersize=size, label=f"{fatalities_bins[i][0]} - {fatalities_bins[i][1]}", alpha=0.5)

axs[1].set_title("Fatalities in Nigeria (2000-2022)")
axs[1].set_xlabel("Longitude")
axs[1].set_ylabel("Latitude")
axs[1].legend(title="Fatalities", loc="lower right")

# Visualize distribution of armed clash and fatalities across the States
plt.show()

### 3F. Seasonal Variations in the Armed Clashes in Nigeria

In [ ]:
# Function to aggregate df by months
def aggregate_by_month(data, ym_columns):
    
    # Aggregate data by year and month
    df = (data
            .groupby(ym_columns).size()
            .reset_index(name = "Count"))

    # Pivot 
    df = (df.pivot(index = ym_columns[0], 
                    columns = ym_columns[1], values="Count")
                    .fillna(0))

    return df

In [ ]:
# Armed clash heatmap (seasonal variation)
# Month and year columns 
acled_year_month_columns = ["year", "month"]

# Monthly aggregate 
spill_monthly_agg = aggregate_by_month(nga_acled_data, acled_year_month_columns)

# Monthly heatmap
nga_heat_title  = "Heatmap of Monthly Armed Clash Events in Nigeria (2000-2022)"
month_heat_map = plot_heatmap(spill_monthly_agg, "Armed Clash", nga_heat_title)

# Save the heatmap
#plt.gcf().savefig("Maps_charts/Heatmap_Armed_Clash_Events_in_Nigeria.png", transparent=True)

### 3G. Armed Clash & Fatalities Analysis in the ROIs

In [ ]:
_ROI_LOCATIONS = ["Maiduguri", "Bama", "Monguno", "Gwoza", "Damboa", "Hadejia"]


def wrangle_acled_roi(
    data: pd.DataFrame,
    rois: list[str] = _ROI_LOCATIONS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Subset ACLED data to specific ROIs and aggregate by location and year.

    Args:
        data: Cleaned ACLED DataFrame produced by wrangle_acled_nga.
        rois: Locations to retain. Defaults to the six primary study sites.

    Returns:
        Tuple of:
            df            – ROI-filtered event-level DataFrame.
            df_total      – Total armed clashes and fatalities per location.
            df_count_year – Annual armed clashes and fatalities per location.
    """

    df = data[data["location"].isin(rois)]

    df_total = (
        df.groupby("location")
        .agg({"sub_event_type": "count", "fatalities": "sum"})
        .reset_index()
        .rename(columns={"sub_event_type": "armed clash"})
        .sort_values("armed clash")
    )
    
    # Events and total fatalities per year
    df_count_year = (
        df.groupby(["year", "location"])
        .agg({"sub_event_type": "count", "fatalities": "sum"})
        .reset_index()
        .rename(columns={"sub_event_type": "armed clash", "location": "name"})
        .fillna(0)
    )

    return df, df_total, df_count_year


# Aggregate armed clash and fatalities in the ROIs
df_armed_roi, df_armed_total_roi, df_armed_year_count_roi = wrangle_acled_roi(nga_acled_data)
#display(df_armed_roi.head())

print("Sum total the number of events and total fatalities in each ROI:")
display(df_armed_total_roi.head())

print("\nNumber of events and total fatalities per year in each ROI:")
display(df_armed_year_count_roi.head())

In [ ]:
# Temporal trend of armed clash in the ROIS
armed_roi_chart_title = "Temporal Trend of Armed Clash Events in the ROIs (2000 - 2022)"
fig_armed_temp_roi = plot_line_chart_roi(
                                        df_armed_year_count_roi,
                                        "year",
                                        "armed clash",
                                        armed_roi_chart_title,
                                        "Year",
                                    )
fig_armed_temp_roi

In [ ]:
# Temporal trend of fatalities in the ROIS
fat_roi_chart_title = "Temporal Trend of Fatalities in the ROIs (2000 - 2022)"
fig_fat_temp_roi = plot_line_chart_roi(df_armed_year_count_roi, 
                                      "year", 
                                      "fatalities",
                                      fat_roi_chart_title, 
                                      "Year") 
fig_fat_temp_roi

In [ ]:
# Pivot yearly armed clash count 
df_armed_year_count_roi_pivot = (
    df_armed_year_count_roi
    .fillna(0)
    .pivot(index="name", columns="year", values="armed clash")
)

print("Pivoted Armed Clashes in ROIs:")
display(df_armed_year_count_roi_pivot.head())

# df_armed_year_count_roi_pivot.to_csv("Outputs/Spreadsheets/ACLED_Armed_Clash_Count_ROI.csv")

# Pivot yearly fatalities
df_fat_year_count_roi_pivot = (
    df_armed_year_count_roi
    .fillna(0)
    .pivot(index="name", columns="year", values="fatalities")
)

print("Pivoted Fatality in ROIs:")
display(df_fat_year_count_roi_pivot.head())

# df_fat_year_count_roi_pivot.to_csv("Outputs/Spreadsheets/ACLED_Fatalities_Count_ROI.csv")

In [ ]:
# Filter Maiduguri armed clash events between 2010 and 2022 (conflict hotspots) 
df_armed_roi_10_22 = df_armed_roi[
    df_armed_roi["year"].between(2010, 2022)
    & (df_armed_roi["location"] == "Maiduguri")
]

# Count monthly events per location and pivot to wide format
df_armed_monthly_roi = (
    df_armed_roi_10_22
    .groupby(["year", "month", "location"]).size()
    .reset_index(name="armed clash")
    .pivot(index="year", columns=["location", "month"], values="armed clash")
    .fillna(0)
)

# Flatten MultiIndex columns to "location_month" strings
df_armed_monthly_roi.columns = [
    f"{loc}_{month}" for loc, month in df_armed_monthly_roi.columns
]

# Sort columns by location then by month and reorder
custom_order = sorted(
    df_armed_monthly_roi.columns,
    key=lambda col: (col.split("_")[0], int(col.split("_")[1])),
)
df_armed_monthly_roi = df_armed_monthly_roi[custom_order]

df_armed_monthly_roi.head()

In [ ]:
# Heat map of monthly trend of armed clashes in the ROIS
roi_heat_title = "Heatmap of Monthly Armed Clash Events Maiduguri (2010-2022)"
plot_heatmap(df_armed_monthly_roi, "Armed Clash", roi_heat_title)

## 4. ROIs Boundary 

This study used GRID3 NGA - Settlement Extents v3.1. However, the data has been updated to [V4.0, as of April 2026](https://data.grid3.org/datasets/GRID3::grid3-nga-settlement-extents-v4-0/about). 

The V3.1 is archived [here](https://academiccommons.columbia.edu/doi/10.7916/x9xg-e262).

In [ ]:
# GRID3 NGA BOUNDARY USING API

grid3_v31_url = "https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_NGA_settlement_extents_v3_1_gdb/FeatureServer/0/query"

offset = 0
record_count = 2000 
max_features = 50 
all_features = []

limit_text = "all features" if max_features is None else f"up to {max_features} features"
print(f"Starting download from {grid3_v31_url} (target: {limit_text})...")

while True:
    current_record_count = record_count
    
    # Max_features limit
    if max_features is not None:
        remaining = max_features - len(all_features)
        if remaining <= 0:
            print("Reached the maximum feature limit.")
            break
        current_record_count = min(record_count, remaining)

    params = {
        "where": "1=1",
        "outFields": "*",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": current_record_count,
        "outSR": "4326" 
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status() 
    except Exception as e:
        print(f"Failed to fetch data at offset {offset}. Error: {e}")
        print("Waiting 5 seconds before retrying...")
        time.sleep(5)
        continue
        
    data = response.json()
    
    if "error" in data:
        print(f"ArcGIS API Error: {data['error'].get('message', data['error'])}")
        break
        
    if "features" not in data:
        print(f"Unexpected response at offset {offset}: {data}")
        break
        
    features = data["features"]
    if not features:
        break
        
    all_features.extend(features)
    print(f"Downloaded {len(all_features)} features so far...")
    
    if len(features) < current_record_count:
        break
        
    offset += current_record_count

print(f"\nFinished downloading. Total settlements: {len(all_features)}")

if all_features:

    gdf = gpd.GeoDataFrame.from_features(all_features)
    gdf.set_crs(epsg=4326, inplace=True)
    
    output_file = "GRID3_NGA_settlement_extents_v3_1.geojson"
    print(f"Saving to {output_file}...")
    gdf.to_file(output_file, driver="GeoJSON")
    print("Save complete!")
    

    display(gdf.head())
else:
    print("No features were downloaded.")


In [ ]:
# GRID3 NGA BOUNDARY USING DOWNLOADED FILE

# FeatureCollection to GeoDataFrame (ROIs)
def fc_to_gdf(
    fc: ee.FeatureCollection,
    filter_columns: list[str],
) -> gpd.GeoDataFrame:
    """Convert an Earth Engine FeatureCollection to a GeoDataFrame."""
    return geemap.ee_to_gdf(fc)[filter_columns]

# Shapefile to FeatureCollection (ROIs)
def shp_to_fc(
    shp_path: str | Path,
    id_column: str,
    id_values: list[str],
) -> ee.FeatureCollection:
    """Load a shapefile and return a filtered Earth Engine FeatureCollection."""
    return geemap.shp_to_ee(shp_path).filter(ee.Filter.inList(id_column, id_values))


# ids of ROIs
grid3_id_column = "mgrs_code"
grid3_mgrs_list = ["33PTP9709_1", "33PUN5774_1", "33PUN5725_1", 
                   "33PTN5533_1", "33PUQ4901_1",  "32PPU1277_1"]
grid3_filter_columns = ["geometry", "mgrs_code", "name"]

roi_fc = shp_to_fc(grid3_settlements_path, grid3_id_column, grid3_mgrs_list)
roi_gdf = fc_to_gdf(roi_fc, grid3_filter_columns)

display(roi_gdf.head(6))

In [ ]:
# Visualize the ROIs 
roi_gdf.explore()

roi_map = feature_map(roi_fc, {"color": "red", 
                                "fillColor": "00000000", 
                                "width": 2}, "ROIs", map_centre_coord)
roi_map

## 5. Land Use Land Cover Change Analysis

### 5A. Helper Functions: Land Cover

In [ ]:
def rename_classes(
    image: ee.Image,
    class_values: list[int],
    new_values: ee.List,
) -> ee.Image:
    """Remap native class values to sequential integers and rename the band."""
    return image.remap(class_values, new_values).rename("classification")

In [ ]:
def landcover_processing(
    image_col: ee.ImageCollection,
    year: int,
    fc: ee.FeatureCollection,
    scale: int = 30,
) -> ee.FeatureCollection:
    """Process land cover data for a specific year and region.

    Groups raster classes into six standard categories and computes the area
    (km²) for each land cover type per ROI.

    Args:
        image_col: ImageCollection containing annual land cover data.
        year: Year to process.
        fc: FeatureCollection of ROIs.
        scale: Pixel resolution in metres for area calculation (default 30).

    Returns:
        Flattened FeatureCollection with land cover class names and area (km²)
        per ROI.
    """

    def group_landcover_raster(feature: ee.Feature) -> ee.Image:
        """Clip the land cover image for the given year and remap to six categories."""

        land_cover = (
            image_col
            .filter(ee.Filter.eq("year", year))
            .first()
            .clip(feature.geometry())
        )

        land_cover = land_cover.set({
            "image_date": int(year),
            "roi_name": feature.get("name"),
        })

        land_cover_mapping = {
            "Cropland":                [1, 2, 4],
            "Forest":                  [5, 6, 7, 8, 9, 13],
            "Shrubland and Grassland": [15, 16, 17, 18, 19, 20, 21, 22],
            "Water body":              [24, 25, 34],
            "Built-up Areas":          [30],
            "Bare Areas":              [31, 32],
        }

        remap_from, remap_to = [], []
        for idx, (_, values) in enumerate(land_cover_mapping.items(), start=1):
            remap_from.extend(values)
            remap_to.extend([idx] * len(values))

        return land_cover.remap(remap_from, remap_to, defaultValue=0)


    def landcover_raster_area(feature: ee.Feature) -> ee.FeatureCollection:
        """Compute area (km²) for each land cover class within a given ROI."""

        land_cover_image = group_landcover_raster(feature)

        banded = ee.Image.pixelArea().divide(1e6).addBands(land_cover_image)

        land_cover_areas = banded.reduceRegion(
            reducer=ee.Reducer.sum().group(1),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e8,
            tileScale=4,
        )

        area_list = ee.List(land_cover_areas.get("groups"))

        land_cover_labels = ee.Dictionary({
            1: "Cropland",
            2: "Forest",
            3: "Shrubland and Grassland",
            4: "Water body",
            5: "Built-up Areas",
            6: "Bare Areas",
        })

        def process_entry(entry) -> ee.Feature:
            """Convert a grouped reducer result into a labelled area Feature."""
            entry = ee.Dictionary(entry)
            return ee.Feature(feature.geometry(), {
                "name": feature.get("name"),
                "year": ee.Number(year).int(),
                "land_cover": land_cover_labels.get(entry.get("group"), "Unknown"),
                "Area_km2": entry.get("sum", 0),
            })

        return ee.FeatureCollection(area_list.map(process_entry))


    land_cover_grouped_images = fc.map(group_landcover_raster)
    land_cover_fc = fc.map(landcover_raster_area)

    print(land_cover_fc.getInfo())

    # Uncomment to export individual land cover layers to Google Drive
    """
    if land_cover_grouped_images.size().getInfo() > 0:
        land_cover_grouped_images_list = land_cover_grouped_images.toList(
            land_cover_grouped_images.size()
        )
        for i in range(land_cover_grouped_images.size().getInfo()):
            land_cover_image = ee.Image(land_cover_grouped_images_list.get(i))
            region_bound = land_cover_image.geometry()
            export_image_to_drive(land_cover_image, 
                                    region_bound, 
                                    "EPSG:32633", 
                                    30, 
                                    "LC", 
                                    out_google_folder)
    else:
        print("No images to export. The Image Collection is empty.")
    """

    return land_cover_fc.flatten()


In [ ]:
def land_cover_change_matrix(
    change_table: str | Path,
    dir_prefix: str,
    file_suffix: str,
) -> None:
    """Convert a land cover change CSV (from SCP QGIS) into a pivot table.

    Computes area transitions between classes in km², appends row and column
    totals, and exports the matrix as a CSV file.

    Args:
        change_table: Path to the input CSV file (from SCP QGIS).
        dir_prefix: Directory and filename prefix for the output CSV.
        file_suffix: Suffix appended to the output filename.
    """

    df = pd.read_csv(change_table)
    df["Area_km2"] = df["Area [metre^2]"] / 1e6

    df = df.pivot_table(
        values="Area_km2",
        index="Reference",
        columns="Classification",
        aggfunc="sum",
        fill_value=0,
    )

    df["Total"] = df.sum(axis=1)
    df.loc["Total"] = df.sum(axis=0)

    df.to_csv(f"{dir_prefix}{file_suffix}.csv")

### 5B. Land Cover Data Processing 

In [ ]:
# GLC_FCS30D Global Land Cover
# Data source : https://gee-community-catalog.org/projects/glc_fcs/?h=glc+fcs30d
# Reference   : https://gee-community-catalog.org/tutorials/examples/glc_fcs30d_lulc/

# Annual land cover ImageCollection, 2000 – 2022
lc_annual = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")

# Classification scheme
# (35 landcover class and 1 fill value)
lc_class_values =  [
  10, 11, 12, 20, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 120, 121, 122, 
  130, 140, 150, 152, 153, 181, 182, 183, 184, 185, 186, 187, 190, 200, 
  201, 202, 210, 220, 0
]

# Land cover class names
lc_class_names = [
    "Rainfed_cropland", "Herbaceous_cover_cropland", "Tree_or_shrub_cover_cropland",
    "Irrigated_cropland", "Open_evergreen_broadleaved_forest", "Closed_evergreen_broadleaved_forest",
    "Open_deciduous_broadleaved_forest", "Closed_deciduous_broadleaved_forest",
    "Open_evergreen_needle_leaved_forest", "Closed_evergreen_needle_leaved_forest",
    "Open_deciduous_needle_leaved_forest", "Closed_deciduous_needle_leaved_forest",
    "Open_mixed_leaf_forest", "Closed_mixed_leaf_forest", "Shrubland",
    "Evergreen_shrubland", "Deciduous_shrubland", "Grassland", "Lichens_and_mosses",
    "Sparse_vegetation", "Sparse_shrubland", "Sparse_herbaceous", "Swamp", "Marsh",
    "Flooded_flat", "Saline", "Mangrove", "Salt_marsh", "Tidal_flat",
    "Impervious_surfaces", "Bare_areas", "Consolidated_bare_areas",
    "Unconsolidated_bare_areas", "Water_body", "Permanent_ice_and_snow", "Filled_value",
]

lc_class_colours = [
    "#ffff64", "#ffff64", "#ffff00", "#aaf0f0", "#4c7300",
    "#006400", "#a8c800", "#00a000", "#005000", "#003c00",
    "#286400", "#285000", "#a0b432", "#788200", "#966400",
    "#964b00", "#966400", "#ffb432", "#ffdcd2", "#ffebaf",
    "#ffd278", "#ffebaf", "#00a884", "#73ffdf", "#9ebb3b",
    "#828282", "#f57ab6", "#66cdab", "#444f89", "#c31400",
    "#fff5d7", "#dcdcdc", "#fff5d7", "#0046c8", "#ffffff", "#ffffff",
]


# Mosaic tiled images and rename bands b1, b2, ... to 2000, 2001, ...
# Each image in annual land cover data has 23 bands, one for each year from 2000-2022 (23 years)
lc_annual_mosaic = lc_annual.mosaic()
years_list = ee.List.sequence(2000, 2022).map(lambda year: ee.Number(year).format("%04d"))
lc_annual_mosaic_renamed = lc_annual_mosaic.rename(years_list)

# Multiband to single-band annual mosaic & assign time and year metadata to each 
lc_yearly_mosaics = years_list.map(
    lambda year: lc_annual_mosaic_renamed
    .select([year])
    .set({
        "system:time_start": ee.Date.fromYMD(ee.Number.parse(year), 1, 1).millis(),
        "system:index": year,
        "year": ee.Number.parse(year),
    })
)

mosaics_col = ee.ImageCollection.fromImages(lc_yearly_mosaics)
print(mosaics_col.first().getInfo())

# Remap native class values to sequential integers
new_class_values = ee.List.sequence(1, ee.List(lc_class_values).length())

In [ ]:
landcover_collection = mosaics_col.map(
    lambda image: rename_classes(image, lc_class_values, new_class_values)
)
print("Pre-processed Land Cover Collection:", landcover_collection.toDictionary().getInfo())

In [ ]:
# 2010 – 2020: peak armed conflict period
land_cover_years_list = list(range(2010, 2021))
# land_cover_years_list = list(range(2000, 2025, 5))  # Five-year intervals

land_cover_area_values_list = [
    landcover_processing(landcover_collection, year, roi_fc, 30)
    for year in land_cover_years_list
]

land_cover_area_values_fc = ee.FeatureCollection(land_cover_area_values_list).flatten()
df_land_cover_area_values = geemap.ee_to_df(land_cover_area_values_fc).fillna(0)

display(df_land_cover_area_values)

# df_land_cover_area_values.to_csv("Outputs/Land_Cover/Spreadsheets/Landcover_area_5Years.csv")
# df_land_cover_area_values.to_csv("Outputs/Land_Cover/Spreadsheets/Landcover_area_2010_2020.csv")

In [ ]:
df_land_cover_area_values_pivot = (
    df_land_cover_area_values
    .pivot_table(index="land_cover", columns=["name", "year"], values="Area_km2")
    .reset_index()
    .fillna(0) # if land cover class is not present for an ROI
)

display(df_land_cover_area_values_pivot.head(7))

# df_land_cover_area_values_pivot.to_csv("Outputs/Land_Cover/Spreadsheets/Landcover_area_5Years_Pivot.csv")
# df_land_cover_area_values_pivot.to_csv("Outputs/Land_Cover/Spreadsheets/Landcover_area_2010_2020_Pivot.csv")

### 5C. Land Cover Change Detection

**Further analysis of land cover change detection was performed using the SCP Plugin in QGIS**

In [ ]:
lc_out_dir_prefix = "Outputs/LC_Change/Transformed/lc_change_matrix_"
lc_change_files = glob(r"Outputs/LC_Change/SCP/lc_Change_*.csv")

for file in lc_change_files:
    suffix = Path(file).stem.split("Change_")[1]
    land_cover_change_matrix(file, lc_out_dir_prefix, suffix)

## 6. Building Fractional Count and Building Presence

In [ ]:
def process_open_building(
    date: int,
    fc: ee.FeatureCollection,
    scale: int,
    epsg: str,
) -> ee.FeatureCollection:
    """Process building presence and count buildings for a given year.

    Uses the Open Buildings 2.5D Temporal Dataset (OBTD) to mosaic building
    data and compute building counts per ROI.

    Args:
        date: Year to process.
        fc: FeatureCollection of ROIs.
        scale: Spatial resolution in metres.
        epsg: Coordinate reference system (e.g. "EPSG:32633").

    Returns:
        FeatureCollection of building count values per ROI.
    """

    def download_open_building(feature: ee.Feature) -> ee.Image:
        """Filter, mosaic, and clip Open Buildings data for the given year and ROI."""

        open_building_collection = (
            ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
            .filterBounds(feature.geometry())
        )

        epoch = ee.Date(f"{date}-06-30", "America/Los_Angeles").millis().divide(1000)

        open_building_mosaic = (
            open_building_collection
            .filter(ee.Filter.eq("inference_time_epoch_s", epoch))
            .mosaic()
            .clip(feature.geometry())
        )

        roi_name = feature.get("name") if isinstance(feature, ee.Feature) else "Unnamed_ROI"
        
        # Add custom image properties
        return open_building_mosaic.set({
            "image_date": int(date),
            "roi_name": roi_name,
        })


    def count_open_building(feature: ee.Feature) -> ee.Feature | None:
        """Count buildings within a ROI and return the result as a Feature."""

        open_building_mosaic = download_open_building(feature)

        if open_building_mosaic is None:
            return None

        count_building = open_building_mosaic.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=feature.geometry(),
            scale=scale,
            crs=epsg,
            maxPixels=1e10,
            bestEffort=True,
        ).get("building_fractional_count")

        if count_building is None:
            return None
        
        # Adjust for original resolution
        count_building = ee.Number(count_building).multiply(ee.Number(scale * 2).pow(2))

        return ee.Feature(feature.geometry(), {
            "name": feature.get("name"),
            "year": int(date),
            "building_count": count_building,
        })


    building_mosaic_images = fc.map(download_open_building)
    building_count_fc = fc.map(count_open_building)

    print(building_mosaic_images.getInfo())

    # Uncomment to export building presence layers to Google Drive
    """
    if building_mosaic_images.size().getInfo() > 0:
        building_mosaic_images_presence = building_mosaic_images.map(
            lambda img: ee.Image(img).select(["building_presence"])
        )
        building_mosaic_images_list = building_mosaic_images_presence.toList(
            building_mosaic_images_presence.size()
        )
        for i in range(building_mosaic_images_presence.size().getInfo()):
            building_presence_image = ee.Image(building_mosaic_images_list.get(i))
            roi_bound = building_presence_image.geometry()
            # export_image_to_drive(building_presence_image, roi_bound, "EPSG:32633", 10, "Building_Presence", out_google_folder)
    else:
        print("No images to export. The Image Collection is empty.")
    """

    return building_count_fc

In [ ]:
# Building data available from 2016 – 2023
open_building_years = range(2016, 2024)

building_count_list = [
    process_open_building(year, roi_fc, 10, "EPSG:32633")
    for year in open_building_years
]

building_count_values_fc = ee.FeatureCollection(building_count_list).flatten()
df_building_counts_values = geemap.ee_to_df(building_count_values_fc).fillna(0)

df_building_counts_values["building_count"] = pd.to_numeric(
    df_building_counts_values["building_count"].apply(lambda x: f"{x:.0f}"),
    errors="coerce",
)


display(df_building_counts_values.head(40))
print(df_building_counts_values.info())

In [ ]:
print(df_building_counts_values.info())

In [ ]:
# Exclude Maiduguri, will make the plot in inbalance
df_building_counts_values_ex = df_building_counts_values[
    ~df_building_counts_values["name"].isin(["Maiduguri"])
]

fig_build_count_bar = go.Figure()

# Separate bars for each year
for year in df_building_counts_values_ex["year"].unique():
    subset = df_building_counts_values_ex[df_building_counts_values_ex["year"] == year]
    fig_build_count_bar.add_trace(go.Bar(
        name=str(year),
        x=subset["name"],
        y=subset["building_count"],
    ))

fig_build_count_bar.update_layout(
    title={
        "text": "<b>Building Counts per ROI (2016 - 2023)</b>",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    title_font=dict(size=18, family="Arial", color="black"),
    xaxis_title="Location",
    yaxis_title="Building Count",
    barmode="group",
    legend_title="Year",
)

fig_build_count_bar.show()

In [ ]:
"""
# Pivot the 'df_building_counts_values' 
df_building_counts_values_pivot = df_building_counts_values.pivot(index="name", 
                                                                columns="year", 
                                                                values="building_count")

df_building_counts_values_pivot.head()

# Export the building count DataFrame as a csv
df_building_counts_values_pivot.to_csv("Outputs/Building_Count/Building_Count.csv")
"""

## 7. Nighttime Lights - GLOBAL-NPP-VIIRS-LIKE-NTL

In [ ]:
def process_nighttime_fc(
    start_date: str,
    end_date: str,
    fc: ee.FeatureCollection,
    scale: int,
) -> ee.FeatureCollection:
    """Compute mean nighttime light intensity per ROI for a given date range.

    Uses VIIRS NTL data. Zero-value pixels are masked before averaging.

    Args:
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        fc: FeatureCollection of ROIs.
        scale: Spatial resolution in metres.

    Returns:
        FeatureCollection with average nighttime light values per feature.
    """

    def download_nighttime(feature: ee.Feature) -> ee.Image:
        """Filter, composite, and clip VIIRS NTL data for the given date range and ROI."""

        return (
            ee.ImageCollection("projects/sat-io/open-datasets/npp-viirs-ntl")
            .filter(ee.Filter.date(start_date, end_date))
            .filterBounds(feature.geometry())
            .mean()
            .clip(feature.geometry())
            .set({
                "image_date": ee.Date(start_date).format("YYYY"),
                "roi_name": feature.get("name"),
            })
        )


    def compute_nighttime_average(feature: ee.Feature) -> ee.Feature:
        """Compute average nighttime light intensity over a ROI."""

        mean_image = download_nighttime(feature)
        masked = mean_image.mask(mean_image.neq(0))  # Exclude zero-value pixels

        stats = masked.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e9,
        )

        return ee.Feature(feature.geometry(), {
            "name": feature.get("name"),
            "year": int(start_date[:4]),
            "average_nighttime": stats.get("b1"),
        })


    mean_nighttime_images = fc.map(download_nighttime)
    average_nighttime_fc = fc.map(compute_nighttime_average)

    # Uncomment to export nighttime composite layers to Google Drive
    """
    if mean_nighttime_images.size().getInfo() > 0:
        mean_nighttime_images_list = mean_nighttime_images.toList(mean_nighttime_images.size())
        for i in range(mean_nighttime_images.size().getInfo()):
            mean_image = ee.Image(mean_nighttime_images_list.get(i))
            region_bound = mean_image.geometry()
            # export_image_to_drive(mean_image, region_bound, "EPSG:32633", 30, "Nighttime", out_google_folder)
    else:
        print("No images to export. The Image Collection is empty.")
    """

    return average_nighttime_fc

In [ ]:
# Period for nighttime analysis
nighttime_years = [(f"{year}-01-01", f"{year}-12-31") 
                    for year in range(2000, 2023)]

nighttime_average_values_list = [
    process_nighttime_fc(start_date, end_date, roi_fc, 500)
    for start_date, end_date in nighttime_years
]

nighttime_average_values_fc = ee.FeatureCollection(nighttime_average_values_list).flatten()
df_nighttime_average_values = geemap.ee_to_df(nighttime_average_values_fc).fillna(0)

display(df_nighttime_average_values)

In [ ]:
# Nighttime average values
fig_nighttime_1 = plot_line_chart_roi(
                                        df_nighttime_average_values,
                                        "year",
                                        "average_nighttime",
                                        "Average Values of Nighttime Lights (2000 - 2022)",
                                        "Year",
                                        axis_y="Mean Nighttime Light",
                                        fill_na=0,
                                    )

In [ ]:
df_nighttime_average_values_pivot = (
    df_nighttime_average_values
    .fillna(0)
    .pivot(index="name", columns="year", values="average_nighttime")
)

df_nighttime_average_values_pivot.head()

#df_nighttime_average_values_pivot.to_csv("Outputs/Nighttime/Average_Nighttime_Values.csv")

## 8. Population Change Patterns

In [ ]:
# Visualization parameters for GPWv411 population DENSITY layer
vis_params_gpw = {
  "min": 0.0,
  "max": 500.0,
  "palette": ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
}


# Visualization parameters for WorldPop population DISTRIBUTION/COUNT layer
vis_params_wpop = {
    "bands" : ["population"],
    "min" : 0.0,
    "max": 100.0,
    "palette" : ["ffffe7", "FFc869", "ffac1d", "e17735", "f2552c", "9f0c21"]
  }

In [ ]:
def process_pop_collection(
    dataset_id: str,
    dataset_band: str,
    col_name: str,
    start_date: str,
    end_date: str,
    fc: ee.FeatureCollection,
    export_prefix: str,
    scale: int,
    country_filter: bool = False,
) -> ee.FeatureCollection:
    """Compute total population per ROI for a given date range.

    Args:
        dataset_id: Earth Engine dataset ID (e.g., WorldPop or GPW).
        dataset_band: Band name containing population values.
        col_name: Output column name for downstream use.
        start_date: Start date in "YYYY-MM-DD" format.
        end_date: End date in "YYYY-MM-DD" format.
        fc: FeatureCollection of ROIs.
        export_prefix: Prefix for naming exported population rasters.
        scale: Spatial resolution in metres.
        country_filter: If True, filters for country code "NGA" (WorldPop only).

    Returns:
        FeatureCollection with total population values per ROI.
    """

    def download_pop_image(feature: ee.Feature) -> ee.Image:
        """Retrieve and clip the population image for the given date range and ROI."""

        pop_collection = (
            ee.ImageCollection(dataset_id)
            .filterDate(start_date, end_date)
            .filterBounds(feature.geometry())
        )

        if country_filter:
            pop_collection = pop_collection.filterMetadata("country", "equals", "NGA")

        pop_layer = pop_collection.first().select(dataset_band)

        return pop_layer.set({
            "image_date": ee.Date(pop_layer.get("system:time_start")).format("YYYY"),
            "roi_name": feature.get("name"),
        }).clip(feature.geometry())


    def compute_pop_sum(feature: ee.Feature) -> ee.Feature:
        """Compute total population within a ROI."""

        pop_layer = download_pop_image(feature)

        stats = pop_layer.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e9,
        )

        return ee.Feature(feature.geometry(), {
            "name": feature.get("name"),
            "year": int(start_date[:4]),
            "sum_pop": stats.get(dataset_band),
        })


    pop_images = fc.map(download_pop_image)
    pop_sum_fc = fc.map(compute_pop_sum)

    # Uncomment to export population layers to Google Drive
    """
    if pop_images.size().getInfo() > 0:
        pop_images_list = pop_images.toList(pop_images.size())
        for i in range(pop_images.size().getInfo()):
            pop_image = ee.Image(pop_images_list.get(i))
            region_bound = pop_image.geometry()
            # export_image_to_drive(pop_image, 
                                    region_bound, 
                                    "EPSG:32633", 
                                    30, 
                                    export_prefix, 
                                    out_google_folder)
    else:
        print("No images to export. The Image Collection is empty.")
    """

    return pop_sum_fc

In [ ]:
def process_batch_of_features(batch_fc: ee.FeatureCollection) -> pd.DataFrame:
    """Convert a FeatureCollection batch to a DataFrame and format the population column."""

    df_batch = geemap.ee_to_df(batch_fc)

    if df_batch.empty:
        print("Batch is empty.")
        return df_batch

    df_batch = df_batch.rename(columns={"sum_pop": "sum_pop_count"})
    df_batch["sum_pop_count"] = df_batch["sum_pop_count"].apply(lambda x: f"{x:.0f}")

    return df_batch

In [ ]:
# WorldPop population count parameters
worldpop_asset = "WorldPop/GP/100m/pop"
worldpop_band = "population"
worldpop_name = "WorldPop"
worldpop_scale = 100
worldpop_prefix = "Pop_Count"

population_years = [(f"{year}-01-01", f"{year}-12-31") for year in range(2000, 2025, 5)]

sum_pop_count_values_list = [
    process_pop_collection(
        worldpop_asset, worldpop_band, worldpop_name,
        start_date, end_date, roi_fc, worldpop_prefix, worldpop_scale,
        country_filter=True,
    )
    for start_date, end_date in population_years
]

# Direct .flatten() exceeded the EE user memory limit, processed in batches instead
pop_count_sum_values_fc = ee.FeatureCollection(sum_pop_count_values_list).flatten()


batch_size = 10
all_df_sum_pop_count_values = []

for i in range(0, pop_count_sum_values_fc.size().getInfo(), batch_size):
    batch_fc = ee.FeatureCollection(pop_count_sum_values_fc.toList(batch_size, i))
    print(f"Processing batch {i // batch_size + 1}")
    all_df_sum_pop_count_values.append(process_batch_of_features(batch_fc))

df_sum_pop_count_values = pd.concat(all_df_sum_pop_count_values, ignore_index=True)
df_sum_pop_count_values["sum_pop_count"] = pd.to_numeric(
    df_sum_pop_count_values["sum_pop_count"], errors="coerce"
)

display(df_sum_pop_count_values)

In [ ]:
print(df_sum_pop_count_values.info())

In [ ]:
# Exclude Maiduguri, will make the plot in inbalance
df_sum_pop_count_values_ex = df_sum_pop_count_values[
    ~df_sum_pop_count_values["name"].isin(["Maiduguri"])
]

fig_pop_count_bar = go.Figure()

for year in df_sum_pop_count_values_ex["year"].unique():
    subset = df_sum_pop_count_values_ex[df_sum_pop_count_values_ex["year"] == year]
    fig_pop_count_bar.add_trace(go.Bar(
        name=str(year),
        x=subset["name"],
        y=subset["sum_pop_count"],
    ))

fig_pop_count_bar.update_layout(
    title={
        "text": "<b>Population Count Per ROI (2000 - 2020)<b>",
        "x": 0.5,
        "xanchor": "center",
        "yanchor": "top",
    },
    title_font=dict(size=18, family="Arial", color="black"),
    xaxis_title="Location",
    yaxis_title="Population Count",
    barmode="group",
    legend_title="Year",
)

fig_pop_count_bar.show()

In [ ]:
df_sum_pop_count_values.info()

In [ ]:
df_sum_pop_count_values_pivot = df_sum_pop_count_values.pivot(index="name", 
                                                              columns="year", 
                                                              values="sum_pop_count")

df_sum_pop_count_values_pivot.head()

# Export the total population Count DataFrame as a csv
#df_sum_pop_count_values_pivot.to_csv("Outputs/Population/Total_Population_Count.csv")

## 9. OSM Road Network

In [ ]:
from shapely.geometry import Polygon
def download_osm_road(geometry: Polygon) -> gpd.GeoDataFrame:
    """Download the OSM road network for a given polygon and reproject to EPSG:32632.

    Args:
        geometry: Boundary polygon used as the spatial filter.

    Returns:
        GeoDataFrame of road geometries reprojected to EPSG:32632.
    """

    roads = ox.graph_from_polygon(polygon=geometry, network_type="all")
    roads_gdf = ox.graph_to_gdfs(roads, nodes=False)
    return roads_gdf.to_crs("EPSG:32632")


roads_list = [download_osm_road(roi.geometry) for _, roi in roi_gdf.iterrows()]

fig, axes = plt.subplots(1, len(roads_list), figsize=(30, 15), squeeze=False)

for ax, roads_gdf in zip(axes[0], roads_list):
    roads_gdf.plot(ax=ax, linewidth=1, color="black")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Road Network")

plt.show()

In [ ]:
def compute_distance_to_road(
    city_gdf: gpd.GeoDataFrame,
    roads_gdf: gpd.GeoDataFrame,
) -> float:
    """Compute the nearest distance from a region's centroid to the nearest road.

    Args:
        city_gdf: Single-row GeoDataFrame (or itertuples row) containing the boundary.
        roads_gdf: GeoDataFrame of road geometries.

    Returns:
        Minimum distance in CRS units from the centroid to the nearest road.
    """

    roi_buffer = city_gdf.geometry
    roads_within = roads_gdf[roads_gdf.intersects(roi_buffer)]
    return roi_buffer.centroid.distance(roads_within.unary_union)


roi_dist_roads = roi_gdf.copy().to_crs("EPSG:32632")

roi_dist_roads["distance_to_roads"] = [
    compute_distance_to_road(roi, roads)
    for roi, roads in zip(roi_dist_roads.itertuples(), roads_list)
]

# roi_dist_roads = roi_dist_roads[["distance_to_roads", "geometry", "name"]]

roi_dist_roads.head()

## 10. Correlation Analysis

In [ ]:
# Correlation: armed clashes, fatalities, and nighttime light (2000 – 2022)
df_armed_year_count_roi_corr_yearly = df_armed_year_count_roi[
    df_armed_year_count_roi["year"].between(2000, 2022)
]
df_nighttime_corr_yearly = df_nighttime_average_values[
    df_nighttime_average_values["year"].between(2000, 2022)
]

df_combined_corr_yearly = (
    df_armed_year_count_roi_corr_yearly
    .merge(df_nighttime_corr_yearly, on=["year", "name"], how="outer")
)
df_combined_corr_yearly[["armed clash", "fatalities"]] = (
    df_combined_corr_yearly[["armed clash", "fatalities"]].fillna(0)
)

df_combined_corr_yearly.head()

# Relevant columns for correlation analysis
corr_yearly_columns = ["armed clash", "fatalities", "average_nighttime"]

for region in df_combined_corr_yearly["name"].unique():
    region_data = df_combined_corr_yearly[df_combined_corr_yearly["name"] == region]
    corr_matrix = region_data[corr_yearly_columns].corr()

    display(corr_matrix)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title(f"Correlation Matrix for {region}", fontsize=16)
    plt.show()


# Correlation test of significance (Pearson)

correlation_yearly_results = {}


for region in df_combined_corr_yearly["name"].unique():
    region_data = df_combined_corr_yearly[df_combined_corr_yearly["name"] == region]
    correlation_yearly_results[region] = {}
    
    # All possible column pairs
    for i, col1 in enumerate(corr_yearly_columns):
        for col2 in corr_yearly_columns[i + 1:]:
            data = region_data[[col1, col2]].dropna()
            if len(data) > 1: # At least two data points
                corr, p_value = stats.pearsonr(data[col1], data[col2])
                correlation_yearly_results[region][(col1, col2)] = (corr, p_value)
            else:
                correlation_yearly_results[region][(col1, col2)] = (None, None)

for region, results in correlation_yearly_results.items():
    print(f"\nCorrelation results for {region}:")
    for (col1, col2), (corr, p) in results.items():
        print(f"  {col1} vs {col2}: Correlation = {corr:.3f}, P-value = {p:.3f}")

In [ ]:
# Correlation between armed clash, fatalities, and buiding count
# Correlation between armed clashes, fatalities, and nighttime
# Filter all the (Geo)DataFrames to only years between 2000 and 2022 (inclusive)
df_armed_year_count_roi_corr_yearly = df_armed_year_count_roi[
    df_armed_year_count_roi["year"].between(2016, 2022)
]
df_nighttime_corr_yearly = df_nighttime_average_values[
    df_nighttime_average_values["year"].between(2016, 2022)
]

df_combined_corr_yearly = (
    df_armed_year_count_roi_corr_yearly
    .merge(df_nighttime_corr_yearly, on=["year", "name"], how="outer")
)
df_combined_corr_yearly[["armed clash", "fatalities"]] = (
    df_combined_corr_yearly[["armed clash", "fatalities"]].fillna(0)
)

df_combined_corr_yearly.head()


corr_yearly_columns = ["armed clash", "fatalities", "average_nighttime"]

for region in df_combined_corr_yearly["name"].unique():
    region_data = df_combined_corr_yearly[df_combined_corr_yearly["name"] == region]
    corr_matrix = region_data[corr_yearly_columns].corr()

    display(corr_matrix)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title(f"Correlation Matrix for {region}", fontsize=16)
    plt.show()


# Correlation test of significance (Pearson)

correlation_yearly_results = {}

for region in df_combined_corr_yearly["name"].unique():
    region_data = df_combined_corr_yearly[df_combined_corr_yearly["name"] == region]
    correlation_yearly_results[region] = {}

    for i, col1 in enumerate(corr_yearly_columns):
        for col2 in corr_yearly_columns[i + 1:]:
            data = region_data[[col1, col2]].dropna()
            if len(data) > 1:
                corr, p_value = stats.pearsonr(data[col1], data[col2])
                correlation_yearly_results[region][(col1, col2)] = (corr, p_value)
            else:
                correlation_yearly_results[region][(col1, col2)] = (None, None)

for region, results in correlation_yearly_results.items():
    print(f"\nCorrelation results for {region}:")
    for (col1, col2), (corr, p) in results.items():
        print(f"  {col1} vs {col2}: Correlation = {corr:.3f}, P-value = {p:.3f}")

## 11. Internally Displaced Persons (IDPs) Census

In [ ]:
def wrangle_idp(
    file_path: str | Path,
    lga_list: list[str],
    idp_columns: list[str],
) -> pd.DataFrame:
    """Load and filter DTM Nigeria IDP data from a multi-sheet Excel file.

    Sheet names follow the pattern "DTM_Nigeria_SS_R<N>" where N is the
    serial number at the end of the filename.

    Args:
        file_path: Path to the DTM Excel file.
        lga_list: LGA names to retain.
        idp_columns: Column names to keep in the output.

    Returns:
        Filtered DataFrame of IDP records for the specified LGAs in Borno State.
    """

    sheet_number = Path(file_path).stem.split("-")[-1]
    sheet_name = f"DTM_Nigeria_SS_R{sheet_number}"

    df = pd.read_excel(file_path, sheet_name=sheet_name)

    if df.columns[0] != "Date Reported":
        df.rename(columns={df.columns[0]: "Date Reported"}, inplace=True)

    df = df.iloc[1:].reset_index(drop=True)  # Row 1 is a metadata description, not data

    df["Date Reported"] = pd.to_datetime(df["Date Reported"])

    df = df[(df["State"] == "BORNO") & df["LGA"].isin(lga_list)]
    df = df[idp_columns]

    print(df["Date Reported"].astype(str).unique())

    return df

In [ ]:
# Primary path for the IDP census spreadsheets
idp_files = glob(r"Data/IDP_Surveys/Round_31_39/dtm-nigeria-site-assessment-round-*.xlsx")

roi_lga_names = ["BAMA", "DAMBOA", "GWOZA", "MAIDUGURI M. C.", "MONGUNO"]

idp_columns = [
    "Date Reported", "DTM Round", "State", "LGA", "SiteID", "Site Name",
    "Longitude", "Latitude", "Site Type", "Site Classification", "Site Area",
    "Number of Households", "Number of Individuals",
    "Occupation of majority of IDPs", "Have IDPs been displaced previously",
    "Reason for displacement of Majority", "Natural Hazard Risks",
]

idp_frames = [wrangle_idp(file, roi_lga_names, idp_columns) for file in idp_files]
df_idps_borno = pd.concat(idp_frames, ignore_index=True)

# DTM round date ranges from the data dictionary
idp_date_mapping = {
    "R31": "15/01/2020 - 15/02/2020",
    "R32": "26/05/2020 - 15/06/2020",
    "R33": "27/07/2020 - 15/08/2020",
    "R34": "16/10/2020 - 06/11/2020",
    "R35": "09/11/2020 - 21/11/2020",
    "R36": "08/02/2021 - 24/02/2021",
    "R37": "19/04/2021 - 09/06/2021",
    "R38": "21/06/2021 - 27/07/2021",
    "R39": "30/08/2021 - 15/10/2021",
}

df_idps_borno["Dictionary Date"] = df_idps_borno["DTM Round"].map(idp_date_mapping)

print(df_idps_borno.info())
display(df_idps_borno)

# LGA where the camps are located
print(f"IDP camps in Borno State span {df_idps_borno['LGA'].nunique()} LGAs.")

#df_idps_borno.to_csv("Outputs/IDP/IDP_Census_R31_39_Cleaned.csv")

In [ ]:
# Group the 'df_Idps' by 'Dictionary Date'
df_idps_date_group = (
    df_idps_borno
    .groupby(["SiteID", "LGA", "Dictionary Date"])["Number of Individuals"]
    .sum()
    .reset_index()
    .pivot(index=["LGA", "SiteID"], columns="Dictionary Date", values="Number of Individuals")
    .fillna(0)
)

idp_site_coordinates = (
    df_idps_borno.groupby("SiteID")[["Site Name", "Longitude", "Latitude"]]
    .first()
    .reset_index()
)

idp_site_lga = (
    df_idps_borno.groupby("SiteID")[["Site Name", "LGA"]]
    .first()
    .reset_index()
)

df_idps_date_group_final = (
    df_idps_date_group
    .merge(idp_site_coordinates, on="SiteID", how="left")
    .merge(idp_site_lga, on="SiteID", how="left", suffixes=("", "_lga"))
)

df_idps_date_group_final.head()

# NOTE: Longitude and latitude were swapped in the source data for a small number
# of IDP camps and corrected manually in QGIS.
# Affected site IDs: ["BO_S342", "BO_S343", "BO_S344", "BO_S345", "BO_S346", "BO_S347", "BO_S348"]

# df_idps_date_group_final.to_csv("Outputs/IDP/IDP_Census_R31_39_Grouped_Site.csv")

In [ ]:
# Visualize the IDP locations
map = folium.Map(location=[df_idps_date_group_final["Latitude"].mean(), 
                           df_idps_date_group_final["Longitude"].mean()], 
                           zoom_start=8)


for _, row in df_idps_date_group_final.iterrows():
    folium.Marker(location=[row["Latitude"], row["Longitude"]]).add_to(map)

#map.save("map.html")  
map  

In [ ]:
raise StopIteration("Stop here to check outputs before continuing")

## 12. Displacement

In [ ]:
# https://dtm.iom.int/data-and-analysis/dtm-api

In [ ]:
# Operations for which DTM data is publicly available
load_dotenv()
DTM_API_KEY = os.getenv("DTM_API_KEY")
dtm_api = DTMApi(subscription_key=DTM_API_KEY)

all_operation_list = dtm_api.get_all_operations()
all_operation_list[all_operation_list["admin0Name"] == "Nigeria"].head()

In [ ]:
# DTM data for Borno State
dtm_borno_data = dtm_api.get_idp_admin2_data(Operation="Lake Chad Basin Crisis", 
                                            CountryName='Nigeria', 
                                            Admin1Name = "Borno",
                                            FromRoundNumber=1, 
                                            ToRoundNumber=50, # Latest round on the DTM website at the time of this work (2025)
                                            to_pandas=True)
dtm_borno_data.head()

In [ ]:
dtm_borno_data["admin2Name"].unique()

In [ ]:
# Local Government Area (LGA)/admin3 where the ROIs are located in
roi_adm2_names = ["Maiduguri", "Bama", "Gwoza", "Damboa", "Monguno",]

# Select IDPs LGA
dtm_borno_roi_data = dtm_borno_data[dtm_borno_data["admin2Name"].isin(roi_adm2_names)]
display(dtm_borno_roi_data.head(10))

In [ ]:
dtm_borno_roi_data_pivot = dtm_borno_roi_data.pivot_table(
                                        index="admin2Name", 
                                        columns="reportingDate", 
                                        values="numPresentIdpInd", 
                                        aggfunc="sum"
                                        )

dtm_borno_roi_data_pivot = dtm_borno_roi_data_pivot.reset_index()  
dtm_borno_roi_data_pivot = dtm_borno_roi_data_pivot.fillna(0)

display(dtm_borno_roi_data_pivot.head())
#dtm_borno_roi_data_pivot.to_csv("Outputs/IDP/DTM_IDP_LGA.csv")